# Train AIT valence directions from the last prompt token

This Colab notebook trains mean difference, logistic regression, and one-dimensional DAS directions from **AIT last-token activations** for GPT-2 Small and Qwen3-0.6B Base. It invokes the package CLI with `--last-token`, sweeps every non-embedding residual boundary, selects DAS epochs on the AIT eval role, and selects one layer per model and method on AIT test-role logit-flip percent.

DAS trains at the last token but performs checkpoint validation and layer-selection evaluation with all-token patching, matching the sentiment-position protocol. All direction checkpoints, CSVs, provenance, similarity tables, and plots are saved directly to a timestamped Google Drive directory. The AIT test role is used for layer selection and is therefore not an unbiased final evaluation set.

## Before running

1. Choose a Colab GPU runtime. An A100 or L4 is preferable for the two-model sweep.
2. Have a Hugging Face token with read access to the private AIT repository and gated model repositories.
3. Leave `RESUME_RUN_ID = None` for a new date-stamped run. After a disconnection, set it to the existing run directory name to reuse compatible checkpoints.
4. The token is entered through a hidden prompt, passed only in the training subprocess environment, and removed immediately after training.

## 1. User settings

In [ ]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional immutable commit or tag.

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = None  # Example: "2026-09-23_14-30_CDT"

DEVICE = "cuda"
DTYPE = "auto"
MODEL_NAMES = [
    "gpt2-small",
    "qwen-0.6b",
]
METHODS = ["mean_diff", "logistic_regression", "das"]
RUN_EXPERIMENT = True

## 2. Install and verify the project

The notebook uses the reusable experiment, persistence, and reporting APIs. It does not reimplement activation extraction, fitting, causal evaluation, similarity calculation, or artifact serialization.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
imported_root = Path(sentiment_geometry.__file__).resolve().parent
expected_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
if imported_root != expected_root:
    raise ImportError(f"Imported {imported_root}, expected {expected_root}")
print("Project commit:", project_commit)
print("Package root:  ", imported_root)

## 3. Mount Google Drive and create today's run

In [ ]:
import torch

from sentiment_geometry.persistence import maybe_mount_google_drive, prepare_timestamped_run

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")

maybe_mount_google_drive(True)
if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name="ait-last-token-directions",
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )

print("Run ID:             ", RUN_LAYOUT.run_id)
print("Run root:           ", RUN_LAYOUT.root)
print("Result CSVs:        ", RUN_LAYOUT.results_dir)
print("Direction artifacts:", RUN_LAYOUT.directions_dir)
print("Figures:            ", RUN_LAYOUT.figures_dir)
if torch.cuda.is_available():
    print("GPU:                ", torch.cuda.get_device_name(0))

## 4. Authenticate to Hugging Face

The token value is never printed, saved in the notebook, written to Drive, or placed in a command argument.

In [ ]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): " ).strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def clear_hf_credentials():
    os.environ.pop("HF_TOKEN", None)
    value = _RUNTIME_SECRETS.pop("HF_TOKEN", None)
    if value is not None:
        del value
    reset_sessions()
    gc.collect()

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
finally:
    del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

## 5. Inspect and lock the experiment contract

The CLI overrides the YAML to use GPT-2 Small and Qwen 0.6B, all three methods, all non-embedding layers, and the last-token representation. Existing per-model batch sizes are retained.

In [ ]:
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from sentiment_geometry.experiments import AITValenceExperimentConfig
from sentiment_geometry.persistence import RunArtifactStore

CONFIG_PATH = PROJECT_ROOT / "configs/ait_valence_directions.yaml"
base_config = AITValenceExperimentConfig.load(CONFIG_PATH)
configured_model_names = [model.name for model in base_config.models]
missing_models = [name for name in MODEL_NAMES if name not in configured_model_names]
if missing_models:
    raise RuntimeError(
        f"Requested models are absent from the config: {missing_models}; "
        f"configured models are {configured_model_names}"
    )
if base_config.data.revision is None:
    raise RuntimeError("The private AIT dataset revision must be pinned.")
if base_config.sampling.train_examples != 55:
    raise RuntimeError("Expected exactly 55 AIT training examples.")
if base_config.sampling.eval_directed_cases != 30:
    raise RuntimeError("Expected exactly 30 AIT eval directed cases.")
if base_config.sampling.test_directed_cases != 30:
    raise RuntimeError("Expected exactly 30 AIT test directed cases.")

contract = {
    "activation_representation": "last_token",
    "fit_position": "final",
    "methods": METHODS,
    "layers": "all_non_embedding",
    "das_training_position": "final",
    "das_checkpoint_validation_position": "all",
    "das_layer_selection_position": "all",
    "das_checkpoint_role": "eval",
    "layer_selection_role": "test",
    "layer_selection_metric": "logit_flip_percent",
    "test_is_unbiased_final_evaluation": False,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("notebook_contract.json", contract)
display(pd.DataFrame([asdict(model) for model in base_config.models]))
display(pd.Series(contract, name="value").to_frame())

## 6. Record the software environment

In [ ]:
import platform
from importlib.metadata import version

environment = {
    "project_commit": project_commit,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": version("torch"),
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("environment.json", environment)
display(pd.Series(environment, name="value").to_frame())

## 7. Train all last-token AIT directions

This cell invokes the real `--last-token` CLI flag. The token is included only in the child process environment and is removed in `finally`, including after an error or interruption. Compatible layer checkpoints are resumed automatically.

In [ ]:
import shutil

cli_path = shutil.which("sentiment-geometry")
if cli_path is None:
    raise RuntimeError("The sentiment-geometry CLI was not installed.")

command = [
    cli_path,
    "train-ait-valence",
    "--config", str(CONFIG_PATH),
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--output-dir", str(RUN_LAYOUT.results_dir),
    "--checkpoint-dir", str(RUN_LAYOUT.directions_dir),
    "--hf-token-env", "HF_TOKEN",
    "--last-token",
    "--all-non-embedding-layers",
]
for model_name in MODEL_NAMES:
    command.extend(["--model", model_name])
for method in METHODS:
    command.extend(["--method", method])

if RUN_EXPERIMENT:
    child_environment = os.environ.copy()
    child_environment["HF_TOKEN"] = get_runtime_secret("HF_TOKEN")
    RUN_LAYOUT.update_manifest(
        status="running",
        metadata={"project_commit": project_commit, "representation": "last_token"},
    )
    try:
        subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            env=child_environment,
            check=True,
        )
        RUN_LAYOUT.update_manifest(status="trained")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        child_environment.pop("HF_TOKEN", None)
        del child_environment
        clear_hf_credentials()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    clear_hf_credentials()
    print("RUN_EXPERIMENT is False; loading an existing run only.")

## 8. Validate and display selected layers and metrics

In [ ]:
RESULTS_DIR = RUN_LAYOUT.results_dir
selection = pd.read_csv(RESULTS_DIR / "all_models_layer_selection.csv")
selected_metrics = pd.read_csv(RESULTS_DIR / "all_models_selected_metrics.csv")
dataset_summary = pd.read_csv(RESULTS_DIR / "dataset_summary.csv")

expected_cells = len(MODEL_NAMES) * len(METHODS)
if len(selection) != expected_cells:
    raise RuntimeError(f"Expected {expected_cells} selected layers; found {len(selection)}")
if set(selection["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("A layer was not selected by logit_flip_percent.")
if set(selection["fit_position"]) != {"final"}:
    raise RuntimeError("A selected direction was not fitted at the final token.")

layer_table = selection.pivot(index="model", columns="method", values="selected_layer")
logit_flip_table = selection.pivot(
    index="model", columns="method", values="selection_value_percent"
).round(3)
sign_flip_table = selected_metrics.pivot(
    index="model", columns="method", values="sign_flip_percent"
).round(3)
display(dataset_summary)
display(layer_table.style.set_caption("AIT-selected residual boundary"))
display(logit_flip_table.style.set_caption("Selected-layer logit flip percent"))
display(sign_flip_table.style.set_caption("Selected-layer sign flip percent"))

## 9. First, middle, and last layer direction similarity

Absolute cosine is the primary orientation-invariant comparison. Signed cosine remains in the full saved CSV for auditing orientation.

In [ ]:
similarities = pd.read_csv(RESULTS_DIR / "all_models_direction_similarities.csv")
method_rank = {method: index for index, method in enumerate(METHODS)}
similarities["method_a_rank"] = similarities["method_a"].map(method_rank)
similarities["method_b_rank"] = similarities["method_b"].map(method_rank)
similarity_summary = similarities[
    similarities["method_a_rank"] < similarities["method_b_rank"]
].copy()

role_rows = []
for model_name, model_rows in similarity_summary.groupby("model", sort=False):
    boundaries = sorted(model_rows["layer"].unique())
    if len(boundaries) != 3:
        raise RuntimeError(
            f"Expected first/middle/last boundaries for {model_name}; got {boundaries}"
        )
    roles = dict(zip(boundaries, ["first", "middle", "last"]))
    selected = model_rows.copy()
    selected["boundary_role"] = selected["layer"].map(roles)
    role_rows.append(selected)
similarity_summary = pd.concat(role_rows, ignore_index=True)
similarity_summary["method_pair"] = (
    similarity_summary["method_a"] + " vs " + similarity_summary["method_b"]
)
summary_columns = [
    "model", "boundary_role", "layer", "method_a", "method_b",
    "signed_cosine", "absolute_cosine",
]
summary_path = RESULTS_DIR / "snapshot_similarity_summary.csv"
similarity_summary[summary_columns].to_csv(summary_path, index=False)
similarity_table = similarity_summary.pivot(
    index=["model", "boundary_role", "layer"],
    columns="method_pair",
    values="absolute_cosine",
).round(3)
display(similarity_table.style.set_caption("Absolute cosine at first/middle/last boundaries"))
print("Saved compact similarity report:", summary_path)

## 10. Save and display plots

In [ ]:
from IPython.display import Image, display

from sentiment_geometry.reporting import plot_ait_valence_run

figure_paths = plot_ait_valence_run(
    RESULTS_DIR, figure_dir=RUN_LAYOUT.figures_dir
)
figure_manifest = pd.DataFrame(
    {
        "figure": [path.name for path in figure_paths],
        "path": [str(path) for path in figure_paths],
    }
)
figure_manifest.to_csv(RESULTS_DIR / "figure_manifest.csv", index=False)
display(figure_manifest)
for path in figure_paths:
    display(Image(filename=str(path)))

## 11. Artifact and credential audit

In [ ]:
required_result_files = [
    "requested_config.json",
    "sample_manifest.csv",
    "pair_manifest.csv",
    "dataset_summary.csv",
    "all_models_metrics.csv",
    "all_models_patching_records.csv",
    "all_models_direction_metadata.csv",
    "all_models_das_epoch_metrics.csv",
    "all_models_direction_similarities.csv",
    "all_models_layer_selection.csv",
    "all_models_selected_metrics.csv",
    "snapshot_similarity_summary.csv",
    "figure_manifest.csv",
    "experiment_manifest.json",
]
missing = [name for name in required_result_files if not (RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing required artifacts: {missing}")

direction_metadata = pd.read_csv(RESULTS_DIR / "all_models_direction_metadata.csv")
checkpoint_files = list(RUN_LAYOUT.directions_dir.rglob("*.npz"))
if len(checkpoint_files) < len(direction_metadata):
    raise RuntimeError(
        f"Expected at least {len(direction_metadata)} checkpoints; found {len(checkpoint_files)}"
    )
if os.environ.get("HF_TOKEN") is not None:
    raise RuntimeError("HF_TOKEN remains in the notebook environment.")
if "HF_TOKEN" in _RUNTIME_SECRETS:
    raise RuntimeError("HF_TOKEN remains in the notebook secret cache.")

RUN_LAYOUT.update_manifest(
    status="completed",
    metadata={
        "models": MODEL_NAMES,
        "methods": METHODS,
        "representation": "last_token",
        "direction_checkpoints": len(checkpoint_files),
        "direction_rows": len(direction_metadata),
        "similarity_rows": len(similarities),
        "figures": len(figure_paths),
    },
)
print("Run completed and audited:", RUN_LAYOUT.root)
print("Verified: HF_TOKEN is absent from the environment and notebook secret cache.")